In [ ]:
import altair as alt
import pandas as pd
from birds.source_data.nabbp import LookupTables

import umap
from sklearn.preprocessing import StandardScaler

In [62]:
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [49]:
lookup = LookupTables()

In [50]:
df= lookup.species

df = df[df['ENDANGERED'] == 'Y']
df.head()

,SPECIES_ID,SPECIES_NAME,ALPHA_CODE,TAXONOMIC_ORDER,SCI_NAME,RECOMENDSIZE,ALLOWABLESIZE,ENDANGERED,RAPTOR,GAMEBIRD
24,230,Marbled Murrelet,MAMU,391.0,Brachyramphus marmoratus,"3, 3B","3, 3B",Y,NaN,NaN
79,720,Roseate Tern,ROST,455.0,Sterna dougallii,"2, 2A","2, 2A, 3",Y,NaN,NaN
86,743,California Least Tern,CLTE,449.0,Sternula antillarum browni,"1A, 1B","1A, 1B, 1P",Y,NaN,NaN
100,820,Short-tailed Albatross,STAL,131.0,Phoebastria albatrus,8,8,Y,NaN,NaN
115,932,Newell's Shearwater,NESH,172.0,Puffinus newelli,"4, 4A","4, 4A",Y,NaN,NaN


In [51]:
# Downloaded from https://opentraits.org/datasets/avonet
# https://figshare.com/s/b990722d72a26b5bfead
DATA_PATH = '/workspaces/mads-siads-696-fall-2025-birds-at-risk/notebooks/data/source_data/ELEData/TraitData/AVONET3_BirdTree.xlsx'


bird_tree_df = pd.read_excel(DATA_PATH,  sheet_name='AVONET3_BirdTree')

bird_tree_df.head()

,Species3,Family3,Order3,Total.individuals,Female,Male,Unknown,Complete.measures,Beak.Length_Culmen,Beak.Length_Nares,...,Migration,Trophic.Level,Trophic.Niche,Primary.Lifestyle,Min.Latitude,Max.Latitude,Centroid.Latitude,Centroid.Longitude,Range.Size,Species.Status
0,Accipiter albogularis,Accipitridae,Accipitriformes,5,2,0,3,4,27.7,17.8,...,2.0,Carnivore,Vertivore,Insessorial,-11.73,-4.02,-8.15,158.493765,37461.21,Extant
1,Accipiter badius,Accipitridae,Accipitriformes,10,4,6,0,8,20.6,12.1,...,3.0,Carnivore,Vertivore,Insessorial,-29.47,46.39,8.23,44.982464,22374973.00,Extant
2,Accipiter bicolor,Accipitridae,Accipitriformes,6,2,2,2,4,26.5,14.8,...,2.0,Carnivore,Vertivore,Generalist,NaN,NaN,NaN,NaN,NaN,Extant
3,Accipiter brachyurus,Accipitridae,Accipitriformes,4,4,0,0,3,22.5,14.0,...,2.0,Carnivore,Vertivore,Insessorial,-6.31,-4.08,-5.45,150.681314,35580.71,Extant
4,Accipiter brevipes,Accipitridae,Accipitriformes,8,4,4,0,4,21.1,12.1,...,3.0,Carnivore,Vertivore,Generalist,31.19,55.86,45.24,45.327340,2936751.80,Extant


In [52]:
bird_tree_df.columns

Index(['Species3', 'Family3', 'Order3', 'Total.individuals', 'Female', 'Male',
       'Unknown', 'Complete.measures', 'Beak.Length_Culmen',
       'Beak.Length_Nares', 'Beak.Width', 'Beak.Depth', 'Tarsus.Length',
       'Wing.Length', 'Kipps.Distance', 'Secondary1', 'Hand-Wing.Index',
       'Tail.Length', 'Mass', 'Mass.Source', 'Mass.Refs.Other', 'Inference',
       'Traits.inferred', 'Reference.species', 'Habitat', 'Habitat.Density',
       'Migration', 'Trophic.Level', 'Trophic.Niche', 'Primary.Lifestyle',
       'Min.Latitude', 'Max.Latitude', 'Centroid.Latitude',
       'Centroid.Longitude', 'Range.Size', 'Species.Status'],
      dtype='object')

In [53]:
bt= set(bird_tree_df['Species3'].unique())
d = set(df['SCI_NAME'].unique())

overlap  = bt.intersection(d)
len(overlap)

43

In [54]:
merged_data = bird_tree_df.join(df.set_index('SCI_NAME').add_prefix('nabbp_'), on='Species3', how='left')
merged_data.head()

,Species3,Family3,Order3,Total.individuals,Female,Male,Unknown,Complete.measures,Beak.Length_Culmen,Beak.Length_Nares,...,Species.Status,nabbp_SPECIES_ID,nabbp_SPECIES_NAME,nabbp_ALPHA_CODE,nabbp_TAXONOMIC_ORDER,nabbp_RECOMENDSIZE,nabbp_ALLOWABLESIZE,nabbp_ENDANGERED,nabbp_RAPTOR,nabbp_GAMEBIRD
0,Accipiter albogularis,Accipitridae,Accipitriformes,5,2,0,3,4,27.7,17.8,...,Extant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Accipiter badius,Accipitridae,Accipitriformes,10,4,6,0,8,20.6,12.1,...,Extant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Accipiter bicolor,Accipitridae,Accipitriformes,6,2,2,2,4,26.5,14.8,...,Extant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Accipiter brachyurus,Accipitridae,Accipitriformes,4,4,0,0,3,22.5,14.0,...,Extant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Accipiter brevipes,Accipitridae,Accipitriformes,8,4,4,0,4,21.1,12.1,...,Extant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
merged_data.columns

Index(['Species3', 'Family3', 'Order3', 'Total.individuals', 'Female', 'Male',
       'Unknown', 'Complete.measures', 'Beak.Length_Culmen',
       'Beak.Length_Nares', 'Beak.Width', 'Beak.Depth', 'Tarsus.Length',
       'Wing.Length', 'Kipps.Distance', 'Secondary1', 'Hand-Wing.Index',
       'Tail.Length', 'Mass', 'Mass.Source', 'Mass.Refs.Other', 'Inference',
       'Traits.inferred', 'Reference.species', 'Habitat', 'Habitat.Density',
       'Migration', 'Trophic.Level', 'Trophic.Niche', 'Primary.Lifestyle',
       'Min.Latitude', 'Max.Latitude', 'Centroid.Latitude',
       'Centroid.Longitude', 'Range.Size', 'Species.Status',
       'nabbp_SPECIES_ID', 'nabbp_SPECIES_NAME', 'nabbp_ALPHA_CODE',
       'nabbp_TAXONOMIC_ORDER', 'nabbp_RECOMENDSIZE', 'nabbp_ALLOWABLESIZE',
       'nabbp_ENDANGERED', 'nabbp_RAPTOR', 'nabbp_GAMEBIRD'],
      dtype='object')

In [ ]:
columns_to_keep = [
    'Species3', 
    # 'Family3', 
    # 'Order3', 
#  'Total.individuals', 
#  'Female', 
#  'Male', 
#  'Unknown', 
#  'Complete.measures', 
    'Beak.Length_Culmen',
    'Beak.Length_Nares', 
    'Beak.Width', 
    'Beak.Depth',
    'Tarsus.Length',
    'Wing.Length', 
    'Kipps.Distance', 
#  'Secondary1', 
 'Hand-Wing.Index',
 'Tail.Length', 
 'Mass', 
 #'Mass.Source', 
 #'Mass.Refs.Other', 
 #'Inference',
 #'Traits.inferred', 
 #'Reference.species', 
#  'Habitat', 
#  'Habitat.Density',
#  'Migration', 
#  'Trophic.Level', 
#  'Trophic.Niche', 
#  'Primary.Lifestyle',
#  'Min.Latitude', 
#  'Max.Latitude', 
#  'Centroid.Latitude',
# 'Centroid.Longitude', 
# 'Range.Size', 
# 'Species.Status'
'nabbp_ENDANGERED'
]

cleaned = (
    merged_data[columns_to_keep]
        .fillna({
            "nabbp_ENDANGERED": 'N',
    })

)

cleaned.head()

,Species3,Beak.Length_Culmen,Beak.Length_Nares,Beak.Width,Beak.Depth,Tarsus.Length,Wing.Length,Kipps.Distance,Hand-Wing.Index,Tail.Length,Mass,nabbp_ENDANGERED
0,Accipiter albogularis,27.7,17.8,10.6,14.7,62.0,235.2,81.8,33.9,169.0,248.75,N
1,Accipiter badius,20.6,12.1,8.8,11.6,43.0,186.7,62.5,32.9,140.6,131.15,N
2,Accipiter bicolor,26.5,14.8,9.2,13.5,57.5,231.8,46.4,19.8,188.4,287.54,N
3,Accipiter brachyurus,22.5,14.0,8.9,11.9,61.2,202.2,64.1,31.7,140.8,142.00,N
4,Accipiter brevipes,21.1,12.1,8.7,11.1,46.4,217.6,87.8,40.2,153.5,186.48,N


In [57]:
data = cleaned[
    [
    'Beak.Length_Culmen',
    'Beak.Length_Nares', 
    'Beak.Width', 
    'Beak.Depth',
    'Tarsus.Length',
    'Wing.Length', 
    'Kipps.Distance', 
    'Hand-Wing.Index',
    'Tail.Length', 
    'Mass', 
]].values


scaled_data = StandardScaler().fit_transform(data)
labels = cleaned['nabbp_ENDANGERED'].values

In [58]:
reducer = umap.UMAP(
    random_state=42,
    min_dist=0.1,
    n_neighbors=15,
    metric='euclidean'
)

In [59]:
embedding = reducer.fit_transform(scaled_data)

/venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
source = pd.DataFrame({
    'umap_component_1': embedding[:, 0], 
    'umap_component_2': embedding[:, 1], 
    'is_endangered': labels
})

In [ ]:
#umap.plot.points(mapper, labels=cleaned['nabbp_ENDANGERED'].values)

chart = (
    alt.Chart(source)
        .mark_circle(size=30)
        .encode(
            x='umap_component_1',
            y='umap_component_2',
            color='is_endangered',
            tooltip=['is_endangered']
        )
        .interactive()
        .properties(
            title='UMAP Projection of Bird Traits (Endangered vs Not Endangered)',
            width=800,
            height=800
        )
)

chart

alt.Chart(...)